# Parameter Optimization and Statistical analysis

In [1]:
import jax.numpy as jnp
import numpy as np
import optax

from bose_harmonic_jax import (
  log_wavefunction_jax,
  wavefunction_derivative_jax,
  local_energy_jax,
  drift_force_jax,
  HarmonicParams,
)
from vmc_jax import MetropolisJAX
from stats import timeseries_bootstrap
from structs import OptimizationRecord, records_to_markdown
from io_utils import upsert_params

cycles_optimize = 10_000
cycles_sample = 2**20
base_dt = 0.05
diffusion_coefficient = 0.5
optimization_iterations = 200
base_lr = 0.01
bootstrap_samples = 2**12
block_size = 2**10

number_particles = [10, 100, 500]
dim = 3

lr_schedule = optax.warmup_cosine_decay_schedule(
  init_value=0.0,
  peak_value=base_lr,
  warmup_steps=optimization_iterations // 10,
  decay_steps=optimization_iterations,
  end_value=base_lr * 1e-2,
)

optimizer = optax.chain(
  optax.clip_by_global_norm(1.0),
  optax.adam(lr_schedule),
)

parameter_guess = HarmonicParams(alpha=jnp.array(0.7))

records: list[OptimizationRecord] = []
for n in number_particles:
  print(f"Optimizing for {n} particles")

  # Optimize parameters
  simulation = MetropolisJAX[HarmonicParams](n, dim)

  time_step = base_dt / np.sqrt(n)
  energy, parameters = simulation.optimize(
    log_wavefunction_jax,
    wavefunction_derivative_jax,
    local_energy_jax,
    drift_force_jax,
    parameter_guess,
    time_step,
    diffusion_coefficient,
    cycles_optimize,
    optimization_iterations,
    optimizer,
  )

  # Bootstrap analysis
  _, _, _, energies, _ = simulation.sample_importance(
    log_wavefunction_jax,
    local_energy_jax,
    drift_force_jax,
    parameters,
    time_step,
    diffusion_coefficient,
    cycles_sample,
  )

  bootstrap_result = timeseries_bootstrap(
    np.array(energies), np.mean, bootstrap_samples, block_size
  )

  records.append(
    OptimizationRecord(
      n=n,
      alpha=float(parameters.alpha),
      mean=bootstrap_result.statistic / n,
      error=bootstrap_result.standard_error / n,
      bias=bootstrap_result.bias / n,
    )
  )

field_map = {
  "n": ("$N$", None),
  "alpha": (r"$\alpha$", ".4f"),
  "mean": (r"$\braket{E}/N$", ".4f"),
  "error": (r"$\operatorname{var}(E)/N$", ".3e"),
  "bias": (r"$\operatorname{bias}(E)/N$", ".3e"),
}
md_table = records_to_markdown(records, field_map)
print(md_table)

param_store: dict[int, float] = {}
for record in records:
  param_store[record.n] = record.alpha
upsert_params("param_store.json", "harmonic", param_store)


Optimizing for 10 particles
iteration 0/200: E=1.84, grad=2.071e-01, HarmonicParams(alpha=Array(0.7, dtype=float32))
iteration 10/200: E=1.81, grad=1.750e-01, HarmonicParams(alpha=Array(0.67284083, dtype=float32))
iteration 20/200: E=1.67, grad=1.320e-01, HarmonicParams(alpha=Array(0.5983809, dtype=float32))
iteration 30/200: E=1.54, grad=3.617e-02, HarmonicParams(alpha=Array(0.5185003, dtype=float32))
iteration 40/200: E=1.47, grad=3.383e-02, HarmonicParams(alpha=Array(0.47741053, dtype=float32))
iteration 50/200: E=1.47, grad=3.099e-02, HarmonicParams(alpha=Array(0.4794788, dtype=float32))
iteration 60/200: E=1.49, grad=7.284e-03, HarmonicParams(alpha=Array(0.49653503, dtype=float32))
Finished at iteration 65: E=1.50, grad=7.232e-04, HarmonicParams(alpha=Array(0.501594, dtype=float32))
Optimizing for 100 particles
iteration 0/200: E=1.84, grad=4.345e-01, HarmonicParams(alpha=Array(0.7, dtype=float32))
iteration 10/200: E=1.81, grad=3.602e-01, HarmonicParams(alpha=Array(0.6729683, dty